In [9]:
import xgrammar as xgr
import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig

In [11]:
# Get tokenizer info
device="cuda"
model_id = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
config = AutoConfig.from_pretrained(model_id)
# This can be larger than tokenizer.vocab_size due to paddings
full_vocab_size = config.vocab_size
tokenizer_info = xgr.TokenizerInfo.from_huggingface(tokenizer, vocab_size=full_vocab_size)

compiler = xgr.GrammarCompiler(tokenizer_info, max_threads=8)

model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.float32, device_map=device
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
config = AutoConfig.from_pretrained(model_id)

In [12]:
tokenizer_info = xgr.TokenizerInfo.from_huggingface(tokenizer, vocab_size=config.vocab_size)
grammar_compiler = xgr.GrammarCompiler(tokenizer_info)

ebnf_grammar_str = """
root ::= number_list

number_list ::= number (number_list)?

number ::= odd even | even odd

even ::= "0" | "2" | "4" | "6" | "8" | "10" | "12" | "14" | "16" | "18" | "20"

odd ::= "1" | "3" | "5" | "7" | "9" | "11" | "13" | "15" | "17" | "19"

"""
compiled_grammar = compiler.compile_grammar(ebnf_grammar_str)

In [14]:
messages = ["Generate a list of numbers that even and odd numbers are not consective:"]
model_inputs = tokenizer(messages, return_tensors="pt").to(model.device)

In [17]:
xgr_logits_processor = xgr.contrib.hf.LogitsProcessor(compiled_grammar)
generated_ids = model.generate(
    **model_inputs, max_new_tokens=10, logits_processor=[xgr_logits_processor]
)
generated_ids = generated_ids[0][len(model_inputs.input_ids[0]) :]
print([tokenizer.decode(generated_ids[i], skip_special_tokens=True) for i in range(len(generated_ids))])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


['1', '123', '45', '67', '89', '1', '01', '123', '45', '678']
